In [4]:
import pandas as pd

# loading the data
df = pd.read_csv("./data/glass.csv")

# removing exact duplicates
df.drop_duplicates(keep="first", inplace=True)
df

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,1
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,7
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,7
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,7
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,7


In [52]:
import pandas as pd
import numpy as np

from collections import Counter

def suspicious_values(
        feature: pd.Series, method: str = "z-score"
) -> list:
    """This function will return the suspicious values in the feature.

    In this function, we will use the univariate method z-score to identify the outliers.
    The outliers are the values that are more than 3 standard deviations away from the mean.
    We will also use the interquartile range (IQR) to identify the outliers usins method argument.

    :param feature: The feature whose outliers we want to identify.
    :param method: The method to use to identify the outliers. Can be "z-score" or "IQR".
    :return: A list of the indices of the suspicious values.
    """
    if method == "z-score":
        # z-score
        z_score = (feature - feature.mean()) / feature.std()

        # identify the outliers
        outliers = z_score[np.abs(z_score) > 3].index.tolist()
        
    if method == "IQR":
        q1 = feature.quantile(0.25)
        q3 = feature.quantile(0.75)
        iqr = q3 - q1

        # identify the outliers	
        outliers = feature[(feature < q1 - 1.5 * iqr) | (feature > q3 + 1.5 * iqr)].index.tolist()
        
    return outliers

suspicious_list = []
for feature in df.columns[:-1]:
    for suspicious_value in suspicious_values(df[feature]):
        suspicious_list.append(suspicious_value)

sus = Counter(suspicious_list).most_common()
sus
# threshold = 2
# for value in sus:
#     if value[1] >= threshold:
#         print(value[0])

[(106, 5),
 (107, 3),
 (163, 3),
 (112, 2),
 (184, 2),
 (171, 2),
 (172, 2),
 (201, 2),
 (188, 1),
 (105, 1),
 (110, 1),
 (111, 1),
 (131, 1),
 (189, 1),
 (203, 1),
 (207, 1),
 (162, 1),
 (174, 1)]

In [59]:
def get_most_suspicious_values(df: pd.DataFrame, method: str = "z-score", threshold: int = 5) -> list:
    suspicious_list = []
    for feature in df.columns[:-1]:
        for suspicious_value in suspicious_values(df[feature], method):
            suspicious_list.append(suspicious_value)

    most_suspicious_values = Counter(suspicious_list).most_common()
    outliers = []
    for value in most_suspicious_values:
        if value[1] >= threshold:
            outliers.append(value[0])
    return outliers

get_most_suspicious_values(df, method="z-score", threshold=3)

[106, 107, 163]